"For a production deployment involving 10 million records, I wouldn't rely on manually copying data for rollback. Since we use Delta Lake, the preferred approach is to leverage Delta Time Travel or take a Deep Clone before deployment. If the deployment fails due to an incorrect MERGE, DELETE, or data corruption, I first stop downstream jobs, identify the last stable version using DESCRIBE HISTORY, and execute RESTORE TABLE TO VERSION AS OF <version>. This restores the table to its previous consistent state with minimal downtime. After restoration, I validate record counts, business-critical metrics, and data quality checks before resuming dependent pipelines. For major releases or disaster recovery requirements, I also maintain Deep Clone or storage-level backups as an additional recovery mechanism."

In [0]:
%sql
CREATE TABLE healthcare.bronze.claims_backup_20260731
AS
SELECT *
FROM healthcare.bronze.claims;

In [0]:
%sql
DESCRIBE HISTORY healthcare.bronze.claims_NEW;

In [0]:
%sql
--ALTER TABLE healthcare.bronze.claims_new SET TBLPROPERTIES ('delta.enableRowTracking' --= 'true'); 
SELECT * FROM table_changes('healthcare.bronze.claims_new', 0, 2);

In [0]:
%sql
-- CREATE TABLE healthcare.bronze.claims_new 
-- AS SELECT * FROM healthcare.bronze.claims VERSION AS OF 0

-- INSERT INTO healthcare.bronze.claims_new VALUES ("1011","M109","P209","3409","2026-07-31","Approved")

SELECT * FROM healthcare.bronze.claims_new


In [0]:
%sql
RESTORE TABLE healthcare.bronze.claims_NEW 
TO VERSION AS OF 0;
-- or 
-- RESTORE TABLE healthcare.bronze.claims 
-- TO TIMESTAMP AS OF '2026-07-31 09:55:00';

In [0]:
%sql
-- Verify 
SELECT *
FROM healthcare.bronze.claims_new 

In [0]:
%sql
DROP TABLE gold.customer;

In [0]:
%sql
CREATE TABLE gold.customer
AS
SELECT *
FROM backup.customer_backup_20260731;